# 🤝 Collaborative Filtering — Thử Nghiệm & Phân Tích

**IT Job Market Intelligence Platform**

Notebook này thực hiện thử nghiệm toàn bộ pipeline **Lọc Cộng Tác (Collaborative Filtering)**:

| Module | Mô tả |
|--------|-------|
| **Dữ liệu tương tác** | 800 ứng viên tổng hợp × ~22k sự kiện tương tác |
| **Item-to-Item CF** | Cosine Similarity trên ma trận User-Item |
| **Matrix Factorization** | TruncatedSVD 30 chiều tiềm ẩn |
| **Persona CF** | Điểm tương thích cộng đồng theo 8 nhóm chuyên môn IT |

---

## 0. Thiết lập môi trường

In [1]:
import os, sys

# Đưa project root vào sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Import thành công!")

Project root: c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence
✅ Import thành công!


## 1. Nạp & Khám Phá Dữ Liệu Tương Tác

In [2]:
INTERACTIONS_CSV = os.path.join(PROJECT_ROOT, "data", "interactions", "user_interactions.csv")
METADATA_JSON    = os.path.join(PROJECT_ROOT, "data", "interactions", "users_metadata.json")

# Nạp dữ liệu tương tác
df = pd.read_csv(INTERACTIONS_CSV)
print(f"Tổng sự kiện tương tác : {len(df):,}")
print(f"Số ứng viên tổng hợp   : {df['user_id'].nunique():,}")
print(f"Số job_id xuất hiện    : {df['job_id'].nunique():,}")
print(f"\nCác loại hành vi (event_type):")
print(df['event_type'].value_counts())
df.head(10)

Tổng sự kiện tương tác : 22,053
Số ứng viên tổng hợp   : 800
Số job_id xuất hiện    : 883

Các loại hành vi (event_type):


KeyError: 'event_type'

In [ ]:
# Phân bố điểm tương tác (rating)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['rating'], bins=20, color='#2563eb', edgecolor='white', linewidth=0.5)
axes[0].set_title('Phân bố Rating Tương Tác', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Điểm rating')
axes[0].set_ylabel('Số sự kiện')

event_counts = df['event_type'].value_counts()
axes[1].bar(event_counts.index, event_counts.values, color=['#2563eb','#059669','#d97706','#dc2626'])
axes[1].set_title('Phân bố Loại Hành Vi', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Event Type')
axes[1].set_ylabel('Số lượng')

plt.tight_layout()
plt.show()

In [ ]:
# Nạp metadata ứng viên (persona distribution)
import json
with open(METADATA_JSON, 'r', encoding='utf-8') as f:
    users_meta = json.load(f)

meta_df = pd.DataFrame(users_meta)
print(f"Số ứng viên trong metadata: {len(meta_df):,}")

persona_dist = meta_df['persona_id'].value_counts()
print("\nPhân bổ Persona:")
print(persona_dist)

# Vẽ biểu đồ persona
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(persona_dist.index, persona_dist.values, color='#2563eb')
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title('Phân Bổ Nhóm Chuyên Môn (Persona)', fontsize=13, fontweight='bold')
ax.set_xlabel('Số ứng viên')
plt.tight_layout()
plt.show()

## 2. Xây Dựng Ma Trận User-Item

In [ ]:
# Tổng hợp rating theo (user_id, job_id) — lấy max rating nếu nhiều lần tương tác
df_agg = df.groupby(['user_id', 'job_id'])['rating'].max().reset_index()

# Mapping index
users  = df_agg['user_id'].unique()
jobs   = df_agg['job_id'].unique()
user_to_idx = {u: i for i, u in enumerate(users)}
job_to_idx  = {j: i for i, j in enumerate(jobs)}
idx_to_job  = {i: j for j, i in job_to_idx.items()}

n_users = len(users)
n_jobs  = len(jobs)
print(f"Ma trận User-Item: {n_users} × {n_jobs}")

# Xây dựng ma trận thưa (Sparse COO)
rows = df_agg['user_id'].map(user_to_idx).values
cols = df_agg['job_id'].map(job_to_idx).values
data = df_agg['rating'].values.astype(np.float32)

R = sp.csr_matrix((data, (rows, cols)), shape=(n_users, n_jobs))
density = R.nnz / (n_users * n_jobs) * 100
print(f"Số phần tử ≠ 0 : {R.nnz:,}")
print(f"Mật độ ma trận : {density:.4f}%  (Data Sparsity rất cao — đây là điều bình thường với CF)")

In [ ]:
# Trực quan hóa mẫu 100×100 đầu tiên
sample = R[:100, :100].toarray()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sample, aspect='auto', cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=ax, label='Rating')
ax.set_title('Mẫu Ma Trận User-Item (100×100 đầu tiên)', fontsize=13, fontweight='bold')
ax.set_xlabel('Job Index')
ax.set_ylabel('User Index')
plt.tight_layout()
plt.show()

## 3. Item-to-Item Collaborative Filtering

In [ ]:
# Tính ma trận tương đồng Item-to-Item (Cosine Similarity)
# Transpose: mỗi hàng là 1 item, mỗi cột là 1 user
R_item = R.T.toarray()  # shape: (n_jobs, n_users)

print("Đang tính Item-to-Item Cosine Similarity...")
item_sim = cosine_similarity(R_item)  # shape: (n_jobs, n_jobs)
print(f"Ma trận tương đồng: {item_sim.shape}")
print(f"Min: {item_sim.min():.4f}  Max: {item_sim.max():.4f}  Mean: {item_sim.mean():.4f}")

In [ ]:
# Kiểm tra Similar Jobs cho 1 công việc mẫu
def get_similar_jobs_manual(job_id: int, top_k: int = 5):
    """Tìm top_k công việc tương tự job_id bằng Item-to-Item CF."""
    if job_id not in job_to_idx:
        return []
    j_idx = job_to_idx[job_id]
    sims  = item_sim[j_idx].copy()
    sims[j_idx] = -1  # Loại bỏ chính nó
    top_indices = np.argsort(sims)[::-1][:top_k]
    return [(idx_to_job[i], round(float(sims[i]), 4)) for i in top_indices]

# Thử với job_id đầu tiên trong tập
sample_job_id = int(jobs[0])
print(f"Tìm Similar Jobs cho job_id = {sample_job_id}:")
similar = get_similar_jobs_manual(sample_job_id, top_k=5)
for rank, (jid, score) in enumerate(similar, 1):
    print(f"  Top {rank}: job_id={jid}  similarity={score:.4f}")

In [ ]:
# Phân bố điểm tương đồng
sim_flat = item_sim[np.triu_indices(len(jobs), k=1)]
print(f"Số cặp item pair: {len(sim_flat):,}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sim_flat[sim_flat > 0], bins=50, color='#2563eb', edgecolor='white')
ax.set_title('Phân Bố Điểm Item-to-Item Similarity (chỉ > 0)', fontsize=13, fontweight='bold')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Số cặp')
plt.tight_layout()
plt.show()

## 4. Matrix Factorization với TruncatedSVD

In [ ]:
# Chia train / test (80/20) để đánh giá
n_total = len(df_agg)
n_train = int(n_total * 0.8)

df_shuffled = df_agg.sample(frac=1, random_state=42).reset_index(drop=True)
df_train = df_shuffled[:n_train]
df_test  = df_shuffled[n_train:]

print(f"Train set: {len(df_train):,} sự kiện")
print(f"Test set : {len(df_test):,} sự kiện")

# Xây ma trận train
rows_tr = df_train['user_id'].map(user_to_idx).values
cols_tr = df_train['job_id'].map(job_to_idx).values
data_tr = df_train['rating'].values.astype(np.float32)

R_train = sp.csr_matrix((data_tr, (rows_tr, cols_tr)), shape=(n_users, n_jobs))

In [ ]:
# Huấn luyện TruncatedSVD với nhiều giá trị n_components khác nhau
components_list = [10, 20, 30, 50, 80]
explained_vars  = []

print("Thực nghiệm số chiều tiềm ẩn SVD:")
for nc in components_list:
    svd = TruncatedSVD(n_components=nc, random_state=42)
    user_fac = svd.fit_transform(R_train)
    explained = svd.explained_variance_ratio_.sum()
    explained_vars.append(explained)
    print(f"  n_components={nc:3d} → Explained Variance: {explained*100:.2f}%")

# Vẽ đường cong Explained Variance
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(components_list, [v*100 for v in explained_vars], 'o-', color='#2563eb', linewidth=2, markersize=8)
ax.axhline(y=explained_vars[2]*100, color='#dc2626', linestyle='--', alpha=0.7, label=f'n=30 → {explained_vars[2]*100:.1f}%')
ax.set_title('Explained Variance vs Số Chiều Tiềm Ẩn SVD', fontsize=13, fontweight='bold')
ax.set_xlabel('n_components')
ax.set_ylabel('Explained Variance (%)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Huấn luyện SVD với n=30 (lựa chọn tốt nhất theo đồ thị)
N_COMPONENTS = 30
svd_model   = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
user_factors = svd_model.fit_transform(R_train)           # (n_users, n_components)
item_factors = svd_model.components_.T                    # (n_jobs, n_components)

print(f"User factors shape: {user_factors.shape}")
print(f"Item factors shape: {item_factors.shape}")
print(f"Explained variance: {svd_model.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
# Đánh giá trên tập Test: RMSE & MAE
def predict_rating(user_id: int, job_id: int) -> float:
    if user_id not in user_to_idx or job_id not in job_to_idx:
        return 3.0  # fallback
    u_vec = user_factors[user_to_idx[user_id]]
    j_vec = item_factors[job_to_idx[job_id]]
    return float(np.dot(u_vec, j_vec))

y_true, y_pred = [], []
for _, row in df_test.iterrows():
    uid, jid, r_true = int(row['user_id']), int(row['job_id']), float(row['rating'])
    r_pred = predict_rating(uid, jid)
    y_true.append(r_true)
    y_pred.append(r_pred)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
mae  = np.mean(np.abs(y_true - y_pred))

print(f"📊 Kết quả đánh giá TruncatedSVD (n_components={N_COMPONENTS}):")
print(f"   RMSE: {rmse:.4f}")
print(f"   MAE : {mae:.4f}")

In [ ]:
# Biểu đồ True vs Predicted
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_true[:500], y_pred[:500], alpha=0.3, color='#2563eb', s=20)
lims = [min(y_true.min(), y_pred.min()) - 0.5, max(y_true.max(), y_pred.max()) + 0.5]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
ax.set_xlabel('Rating Thực (True)')
ax.set_ylabel('Rating Dự đoán (Predicted)')
ax.set_title(f'True vs Predicted Rating  (RMSE={rmse:.3f}, MAE={mae:.3f})', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Persona Collaborative Filtering

In [ ]:
from src.recommender.synthetic_data import PERSONAS_DEF
from src.recommender.collaborative import get_cf_recommender

# Nạp CF recommender đã lưu (hoặc train nếu chưa có)
print("Đang nạp CollaborativeRecommender...")
cf_rec = get_cf_recommender()
print("✅ Nạp thành công!")
print(f"Số personas: {len(PERSONAS_DEF)}")
for p in PERSONAS_DEF:
    print(f"  [{p['id']}] {p['name']}")

In [ ]:
# Điểm CF cho từng persona
persona_score_stats = []

for p in PERSONAS_DEF:
    pid = p['id']
    scores = cf_rec.get_persona_scores_dict(pid)
    vals   = list(scores.values())
    persona_score_stats.append({
        'persona_id': pid,
        'name': p['name'][:30],
        'n_jobs_scored': len(vals),
        'mean_score': round(np.mean(vals), 4),
        'min_score': round(np.min(vals), 4),
        'max_score': round(np.max(vals), 4),
    })

stats_df = pd.DataFrame(persona_score_stats)
print("\nThống kê điểm CF theo Persona:")
display(stats_df)

In [ ]:
# Top 10 việc làm được CF gợi ý cho persona backend_java
TOP_PERSONA = "backend_java"
top_jobs_cf = cf_rec.get_recommendations_for_persona(persona_id=TOP_PERSONA, top_n=10)

print(f"Top 10 việc làm CF cho persona '{TOP_PERSONA}':")
if top_jobs_cf:
    cf_df = pd.DataFrame(top_jobs_cf)[['job_id', 'title', 'company', 'city', 'seniority_level', 'cf_score']]
    display(cf_df)
else:
    print("Chưa có kết quả (cần kết nối PostgreSQL).")

In [ ]:
# So sánh điểm CF trung bình giữa các Persona
fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#2563eb' if s > stats_df['mean_score'].mean() else '#93c5fd' for s in stats_df['mean_score']]
bars = ax.barh(stats_df['name'], stats_df['mean_score'], color=colors)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax.axvline(stats_df['mean_score'].mean(), color='#dc2626', linestyle='--', label='Mean overall')
ax.set_title('Điểm CF Trung Bình Theo Nhóm Chuyên Môn (Persona)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean CF Score')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Kiểm Tra Module CollaborativeRecommender Tích Hợp

In [ ]:
# Similar Jobs — People Also Applied
from src.recommender.collaborative import get_cf_recommender

cf = get_cf_recommender()

# Lấy job_id mẫu từ dữ liệu tương tác
sample_jid = int(df['job_id'].value_counts().index[0])  # job được tương tác nhiều nhất
print(f"Tìm Similar Jobs cho job_id = {sample_jid} (job được tương tác nhiều nhất):")

similar_jobs = cf.get_similar_jobs(job_id=sample_jid, top_k=5)
if similar_jobs:
    for r, j in enumerate(similar_jobs, 1):
        print(f"  #{r}  job_id={j.get('similar_job_id', '?')}  score={j.get('similarity_score', 0):.4f}  [{j.get('title','?')}]")
else:
    print("  Cần kết nối DB để truy xuất metadata.")

## 7. Tổng Kết

In [ ]:
print("=" * 60)
print("📋 TỔNG KẾT THỰC NGHIỆM COLLABORATIVE FILTERING")
print("=" * 60)
print(f"\n[Dữ liệu]")
print(f"  Số ứng viên      : {n_users:,}")
print(f"  Số việc làm CF   : {n_jobs:,}")
print(f"  Số sự kiện       : {len(df):,}")
print(f"  Mật độ ma trận   : {density:.4f}%")

print(f"\n[Item-to-Item CF]")
nonzero_pairs = np.sum(sim_flat > 0)
print(f"  Cặp similarity > 0: {nonzero_pairs:,} / {len(sim_flat):,}")

print(f"\n[TruncatedSVD - Matrix Factorization]")
print(f"  n_components         : {N_COMPONENTS}")
print(f"  Explained variance   : {svd_model.explained_variance_ratio_.sum()*100:.2f}%")
print(f"  Test RMSE            : {rmse:.4f}")
print(f"  Test MAE             : {mae:.4f}")

print(f"\n[Personas]")
for _, row in stats_df.iterrows():
    print(f"  {row['persona_id']:20s} → mean_score={row['mean_score']:.4f}  max={row['max_score']:.4f}")

print("\n✅ CF Pipeline hoàn chỉnh và sẵn sàng tích hợp vào True Hybrid!")